# Satellite decay screening and thermospheric density experiments

Research notebook. Execute the cells in sequence after placing the three
source files in the configured data directory. This version preserves the
numerical methods of the archived v8 audit; outputs are written to a separate
folder to avoid overwriting the original results.

The SATCAT completeness cutoff requires independent provenance confirmation.
CelesTrak indices may change unless the archived index table is retained.


### Environment setup


In [ ]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

required = {
    "numpy": "2.4.6",
    "pandas": "2.2.3",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
    "matplotlib": "3.10.0",
    "pymsis": "0.12.0",
    "xarray": "2025.12.0",
    "joblib": "1.6.0"
}

additional = {
    "h5netcdf": ">=1.4,<2",
    "h5py": ">=3.12,<4",
    "Pillow": ">=10,<13"
}

install = []
changed_loaded = []
for package, target in required.items():
    try:
        current = version(package)
    except PackageNotFoundError:
        current = None
    if current != target:
        install.append(f"{package}=={target}")
        module = "sklearn" if package == "scikit-learn" else package
        if module in sys.modules:
            changed_loaded.append(package)

for package, specifier in additional.items():
    try:
        current = version(package)
    except PackageNotFoundError:
        current = None
    if current is None:
        install.append(f"{package}{specifier}")

if install:
    print("Installing required packages:", ", ".join(install))
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check", "--quiet", *install
    ])
else:
    print("Required packages already installed.")

if changed_loaded:
    raise RuntimeError(
        "Packages were changed after import: " + ", ".join(changed_loaded)
        + ". Restart the runtime and execute all cells again."
    )

for package, target in required.items():
    current = version(package)
    if current != target:
        raise RuntimeError(
            f"Unexpected version for {package}: {current}; required {target}"
        )

print("Python:", sys.version.split()[0])
print("pymsis:", version("pymsis"))
print("Dependencies verified.")


### Data ingestion


In [ ]:
import os
import sys
import warnings
from pathlib import Path
import hashlib
import platform
import numpy as np
import pandas as pd
import sklearn

raiz_configurada = os.environ.get("SATELLITE_DATA_DIR")
if raiz_configurada:
    raiz_padrao = Path(raiz_configurada)
else:
    try:
        from google.colab import drive
    except ImportError:
        raiz_padrao = Path.cwd() / "data"
    else:
        drive.mount("/content/drive")
        raiz_padrao = Path("/content/drive/MyDrive/Aeroespacial")

diretorio_base = Path(
    os.environ.get("SATELLITE_DATA_DIR", str(raiz_padrao))
).expanduser().resolve()
arquivo_csv = diretorio_base / "active_satellites.csv"
diretorio_referencia = diretorio_base / "revisao_jbcs_2026" / "auditoria_v8"
diretorio_resultados = diretorio_base / "revisao_jbcs_2026" / "reexecucao_publica"

hashes_esperados = {
    "TLE": "b4b3e88a9ffcfd15e793bd1eda7ec4570fc056b1249ca6c758dc5ce65e303f48",
    "SATCAT": "4a79c9b205c72c699d40db2053b617396499787769ac619c5f2da6ab3bd0c960",
    "WACCMX": "fe7cc40f4fe7fb40f13ba7c509fb53daaa018c18744f3863131ba61090dbe93f"
}

if not arquivo_csv.is_file():
    raise FileNotFoundError(f"Arquivo nao encontrado: {arquivo_csv}")

sha256 = hashlib.sha256(arquivo_csv.read_bytes()).hexdigest()
if sha256 != hashes_esperados["TLE"]:
    raise ValueError("The TLE data file does not match the archived input (SHA-256).")

df_original = pd.read_csv(arquivo_csv)

colunas_necessarias = [
    "NORAD_CAT_ID",
    "OBJECT_NAME",
    "EPOCH",
    "MEAN_MOTION",
    "ECCENTRICITY",
    "INCLINATION",
    "BSTAR",
    "RA_OF_ASC_NODE",
    "MEAN_MOTION_DOT"
]

ausentes = sorted(set(colunas_necessarias) - set(df_original.columns))

if ausentes:
    raise ValueError(f"Colunas ausentes: {ausentes}")

if df_original["NORAD_CAT_ID"].duplicated().any():
    raise ValueError("Foram encontrados identificadores NORAD duplicados.")

if len(df_original) != 12810:
    raise ValueError(f"Quantidade inesperada de registros: {len(df_original)}")

diretorio_resultados.mkdir(parents=True, exist_ok=True)

print("Registros:", len(df_original))
print("Variaveis:", len(df_original.columns))
print("Valores ausentes:", int(df_original.isna().sum().sum()))
print("IDs NORAD unicos:", df_original["NORAD_CAT_ID"].nunique())
print("SHA-256:", sha256)
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Resultados:", diretorio_resultados)

display(df_original.head())


### Orbital parameters


In [ ]:
df_orbital = df_original.copy()

colunas_numericas = [
    "MEAN_MOTION",
    "ECCENTRICITY",
    "INCLINATION",
    "BSTAR"
]

df_orbital[colunas_numericas] = df_orbital[colunas_numericas].apply(
    pd.to_numeric, errors="raise"
)

df_orbital["EPOCH"] = pd.to_datetime(
    df_orbital["EPOCH"], utc=True, errors="raise"
)

validos = (
    (df_orbital["MEAN_MOTION"] > 0)
    & df_orbital["ECCENTRICITY"].between(0, 1, inclusive="left")
    & df_orbital["INCLINATION"].between(0, 180)
    & np.isfinite(df_orbital[colunas_numericas].to_numpy()).all(axis=1)
)

if not validos.all():
    raise ValueError(
        f"Registros com parametros orbitais invalidos: {(~validos).sum()}"
    )

MU = 398600.4418
R_TERRA = 6378.137
SEGUNDOS_DIA = 86400.0

n_rad_s = (
    df_orbital["MEAN_MOTION"] * 2 * np.pi / SEGUNDOS_DIA
)

a_km = (MU / n_rad_s**2)**(1.0 / 3.0)

df_orbital["PERIOD_MIN"] = (
    1440.0 / df_orbital["MEAN_MOTION"]
)

df_orbital["SEMI_MAJOR_AXIS_KM"] = a_km

df_orbital["ALTITUDE_KM"] = a_km - R_TERRA

df_orbital["PERIGEE_ALT_KM"] = (
    a_km * (1.0 - df_orbital["ECCENTRICITY"]) - R_TERRA
)

df_orbital["APOGEE_ALT_KM"] = (
    a_km * (1.0 + df_orbital["ECCENTRICITY"]) - R_TERRA
)

df_orbital["ABS_BSTAR"] = df_orbital["BSTAR"].abs()

colunas_calculadas = [
    "PERIOD_MIN",
    "SEMI_MAJOR_AXIS_KM",
    "ALTITUDE_KM",
    "PERIGEE_ALT_KM",
    "APOGEE_ALT_KM",
    "ABS_BSTAR"
]

if not np.isfinite(
    df_orbital[colunas_calculadas].to_numpy()
).all():
    raise ValueError("Foram encontrados resultados nao finitos.")

arquivo_saida = diretorio_resultados / "parametros_orbitais.csv"

df_orbital.to_csv(arquivo_saida, index=False)

print("Registros processados:", len(df_orbital))
print("Perigeu minimo (km):", round(df_orbital["PERIGEE_ALT_KM"].min(), 3))
print("Perigeu maximo (km):", round(df_orbital["PERIGEE_ALT_KM"].max(), 3))
print("Perigeus abaixo de 200 km:", (df_orbital["PERIGEE_ALT_KM"] < 200).sum())
print("Epoca inicial:", df_orbital["EPOCH"].min())
print("Epoca final:", df_orbital["EPOCH"].max())
print("Arquivo salvo:", arquivo_saida)

display(
    df_orbital[
        [
            "OBJECT_NAME",
            "NORAD_CAT_ID",
            "SEMI_MAJOR_AXIS_KM",
            "PERIGEE_ALT_KM",
            "APOGEE_ALT_KM",
            "PERIOD_MIN",
            "ABS_BSTAR"
        ]
    ].head()
)


### SATCAT association


In [ ]:
arquivo_satcat = diretorio_base / "satcat.csv"

if not arquivo_satcat.is_file():
    raise FileNotFoundError(
        f"Catalogo SATCAT nao encontrado: {arquivo_satcat}"
    )

df_satcat = pd.read_csv(arquivo_satcat, low_memory=False)

df_satcat.columns = (
    df_satcat.columns.astype(str).str.strip().str.upper()
)

colunas_obrigatorias = [
    "NORAD_CAT_ID",
    "OWNER",
    "OBJECT_TYPE",
    "DECAY_DATE"
]

ausentes = sorted(
    set(colunas_obrigatorias) - set(df_satcat.columns)
)

if ausentes:
    raise ValueError(f"Colunas ausentes no SATCAT: {ausentes}")

df_satcat["NORAD_CAT_ID"] = pd.to_numeric(
    df_satcat["NORAD_CAT_ID"], errors="coerce"
)

ids_invalidos = int(df_satcat["NORAD_CAT_ID"].isna().sum())

if ids_invalidos:
    raise ValueError(
        f"Identificadores NORAD invalidos no SATCAT: {ids_invalidos}"
    )

df_satcat["NORAD_CAT_ID"] = (
    df_satcat["NORAD_CAT_ID"].astype("int64")
)

duplicados = int(
    df_satcat["NORAD_CAT_ID"].duplicated().sum()
)

if duplicados:
    raise ValueError(
        f"Identificadores NORAD duplicados no SATCAT: {duplicados}"
    )

colunas_metadados = [
    "NORAD_CAT_ID",
    "OWNER",
    "OBJECT_TYPE",
    "OPS_STATUS_CODE",
    "LAUNCH_DATE",
    "DECAY_DATE",
    "OBJECT_NAME"
]

colunas_disponiveis = [
    coluna for coluna in colunas_metadados
    if coluna in df_satcat.columns
]

catalogo = df_satcat[colunas_disponiveis].copy()

catalogo = catalogo.rename(
    columns={
        coluna: f"SATCAT_{coluna}"
        for coluna in colunas_disponiveis
        if coluna != "NORAD_CAT_ID"
    }
)

datas_brutas = catalogo["SATCAT_DECAY_DATE"]

datas_convertidas = pd.to_datetime(
    datas_brutas,
    errors="coerce",
    utc=True,
    format="mixed"
)

datas_invalidas = (
    datas_brutas.notna() & datas_convertidas.isna()
)

if datas_invalidas.any():
    raise ValueError(
        f"Datas de decaimento invalidas: {datas_invalidas.sum()}"
    )

catalogo["SATCAT_DECAY_DATE"] = datas_convertidas

df_associado = df_orbital.merge(
    catalogo,
    on="NORAD_CAT_ID",
    how="left",
    validate="one_to_one",
    indicator="SATCAT_JOIN"
)

associados = int(
    (df_associado["SATCAT_JOIN"] == "both").sum()
)

nao_associados = len(df_associado) - associados

decadencias_registradas = int(
    df_associado["SATCAT_DECAY_DATE"].notna().sum()
)

eventos_anteriores = int((
    df_associado["SATCAT_DECAY_DATE"].notna()
    & (
        df_associado["SATCAT_DECAY_DATE"]
        < df_associado["EPOCH"]
    )
).sum())

mesmo_dia = int((
    df_associado["SATCAT_DECAY_DATE"].dt.normalize()
    == df_associado["EPOCH"].dt.normalize()
).sum())

hash_satcat = hashlib.sha256(
    arquivo_satcat.read_bytes()
).hexdigest()
if hash_satcat != hashes_esperados["SATCAT"]:
    raise ValueError("The SATCAT file does not match the archived input (SHA-256).")

arquivo_saida_satcat = (
    diretorio_resultados / "satcat_associado.csv"
)

df_associado.to_csv(
    arquivo_saida_satcat, index=False
)

print("Registros TLE:", len(df_orbital))
print("Registros SATCAT:", len(df_satcat))
print("Objetos associados:", associados)
print("Objetos nao associados:", nao_associados)
print("Datas de decaimento registradas:", decadencias_registradas)
print("Decaimentos anteriores ao EPOCH:", eventos_anteriores)
print("Decaimentos na mesma data do EPOCH:", mesmo_dia)
print("SHA-256 SATCAT:", hash_satcat)
print("Arquivo salvo:", arquivo_saida_satcat)

display(
    df_associado[
        [
            "NORAD_CAT_ID",
            "OBJECT_NAME",
            "EPOCH",
            "PERIGEE_ALT_KM",
            "SATCAT_OWNER",
            "SATCAT_OBJECT_TYPE",
            "SATCAT_DECAY_DATE",
            "SATCAT_JOIN"
        ]
    ].head(10)
)


### 180-day retrospective outcome


In [ ]:
horizonte_dias = 180

# This cutoff is a provenance assumption, not a verified SATCAT download date.
limite_satcat = os.environ.get("SATCAT_CUTOFF_DATE", "2026-10-31")
data_extracao_satcat = pd.Timestamp(limite_satcat, tz="UTC")
satcat_data_confirmada = False
if not satcat_data_confirmada:
    warnings.warn(
        "SATCAT snapshot provenance has not been verified. "
        "The 180-day non-event labels depend on this assumption.",
        RuntimeWarning
    )

df_eventos = df_associado.copy()

data_tle = df_eventos["EPOCH"].dt.normalize()
data_decaimento = df_eventos["SATCAT_DECAY_DATE"].dt.normalize()

fim_observacao = data_tle + pd.Timedelta(
    days=horizonte_dias
)

associado = df_eventos["SATCAT_JOIN"].eq("both")

janela_completa = (
    fim_observacao <= data_extracao_satcat
)

data_inconsistente = (
    data_decaimento.notna()
    & (data_decaimento <= data_tle)
)

elegivel = (
    associado
    & janela_completa
    & ~data_inconsistente
)

df_coorte = df_eventos.loc[elegivel].copy()

inicio = df_coorte["EPOCH"].dt.normalize()

decaimento = (
    df_coorte["SATCAT_DECAY_DATE"].dt.normalize()
)

dias_ate_decaimento = (
    decaimento - inicio
).dt.days

df_coorte["DAYS_TO_DECAY"] = dias_ate_decaimento

df_coorte["OBSERVED_CATALOG_DECAY"] = (
    dias_ate_decaimento.between(
        1, horizonte_dias, inclusive="both"
    )
).astype(int)

n_eventos = int(
    df_coorte["OBSERVED_CATALOG_DECAY"].sum()
)

n_objetos = len(df_coorte)

taxa_eventos = 100 * n_eventos / n_objetos

arquivo_coorte = (
    diretorio_resultados / "coorte_decaimento_180d.csv"
)

df_coorte.to_csv(arquivo_coorte, index=False)

print("Registros iniciais:", len(df_eventos))
print("Sem associacao SATCAT:", int((~associado).sum()))
print("Sem janela completa:", int((~janela_completa).sum()))
print("Datas inconsistentes:", int(data_inconsistente.sum()))
print("Objetos elegiveis:", n_objetos)
print("Eventos em 180 dias:", n_eventos)
print("Sem evento em 180 dias:", n_objetos - n_eventos)
print("Frequencia de eventos (%):", round(taxa_eventos, 4))
print("Data de extracao SATCAT:", data_extracao_satcat.date())
print("Arquivo salvo:", arquivo_coorte)

if n_objetos != 12806 or n_eventos != 403:
    print("ATENCAO: os resultados diferem da analise anterior.")

print("\nRegistros excluidos por inconsistencia temporal:")

display(
    df_eventos.loc[
        data_inconsistente,
        [
            "NORAD_CAT_ID",
            "OBJECT_NAME",
            "EPOCH",
            "SATCAT_DECAY_DATE",
            "PERIGEE_ALT_KM"
        ]
    ]
)

print("\nDistribuicao do desfecho:")

display(
    df_coorte["OBSERVED_CATALOG_DECAY"]
    .value_counts()
    .rename_axis("Evento")
    .reset_index(name="Quantidade")
)


### Sample profile


In [ ]:
df_analise = df_coorte.copy()

limites = [-np.inf, 200, 400, 600, 800, 1200, np.inf]

faixas = [
    "<200 km",
    "200-400 km",
    "400-600 km",
    "600-800 km",
    "800-1200 km",
    ">=1200 km"
]

df_analise["FAIXA_PERIGEU"] = pd.cut(
    df_analise["PERIGEE_ALT_KM"],
    bins=limites,
    labels=faixas,
    right=False
)

nomes = df_analise["OBJECT_NAME"].fillna("").str.upper()

condicoes = [
    nomes.str.startswith("STARLINK"),
    nomes.str.startswith("ONEWEB"),
    nomes.str.startswith("IRIDIUM")
]

df_analise["GRUPO_CONSTELACAO"] = np.select(
    condicoes,
    ["Starlink", "OneWeb", "Iridium"],
    default="Outros"
)

def resumir_eventos(dados, agrupamento):
    tabela = (
        dados.groupby(agrupamento, observed=True)
        ["OBSERVED_CATALOG_DECAY"]
        .agg(total="size", eventos="sum")
        .reset_index()
    )

    tabela["taxa_percentual"] = (
        100 * tabela["eventos"] / tabela["total"]
    )

    return tabela

tabela_altitude = resumir_eventos(
    df_analise, "FAIXA_PERIGEU"
)

tabela_constelacoes = resumir_eventos(
    df_analise, "GRUPO_CONSTELACAO"
)

variaveis_orbitais = [
    "PERIGEE_ALT_KM",
    "APOGEE_ALT_KM",
    "INCLINATION",
    "ECCENTRICITY",
    "ABS_BSTAR"
]

estatisticas_orbitais = (
    df_analise[variaveis_orbitais]
    .agg(["min", "median", "max"])
    .T
    .reset_index()
    .rename(columns={"index": "variavel"})
)

assert tabela_altitude["total"].sum() == 12806
assert tabela_altitude["eventos"].sum() == 403
assert tabela_constelacoes["total"].sum() == 12806
assert tabela_constelacoes["eventos"].sum() == 403

tabela_altitude.to_csv(
    diretorio_resultados / "eventos_por_altitude_v8.csv",
    index=False
)

tabela_constelacoes.to_csv(
    diretorio_resultados / "eventos_por_constelacao_v8.csv",
    index=False
)

estatisticas_orbitais.to_csv(
    diretorio_resultados / "estatisticas_orbitais_v8.csv",
    index=False
)

print("Eventos por altitude do perigeu")
display(tabela_altitude)

print("\nEventos por constelacao")
display(tabela_constelacoes)

print("\nEstatisticas orbitais")
display(estatisticas_orbitais)


### Features and labels


In [ ]:
variaveis_modelo = [
    "PERIGEE_ALT_KM",
    "ABS_BSTAR",
    "INCLINATION",
    "ECCENTRICITY",
    "MEAN_MOTION"
]

variavel_resposta = "OBSERVED_CATALOG_DECAY"

X = df_analise[variaveis_modelo].copy()
y = df_analise[variavel_resposta].astype(int).copy()

ids = df_analise["NORAD_CAT_ID"].copy()
grupos = df_analise["GRUPO_CONSTELACAO"].copy()

if X.isna().any().any():
    raise ValueError("Existem valores ausentes nas variaveis preditoras.")

if not np.isfinite(X.to_numpy(dtype=float)).all():
    raise ValueError("Existem valores nao finitos nas variaveis preditoras.")

if not set(y.unique()).issubset({0, 1}):
    raise ValueError("A variavel resposta deve ser binaria.")

if ids.duplicated().any():
    raise ValueError("Existem identificadores NORAD duplicados.")

if len(X) != len(y) or len(X) != len(ids):
    raise ValueError("Inconsistencia entre preditores e desfechos.")

resumo_variaveis = X.describe().T[
    ["min", "25%", "50%", "75%", "max"]
]

print("Objetos:", len(X))
print("Variaveis preditoras:", X.shape[1])
print("Eventos positivos:", int(y.sum()))
print("Eventos negativos:", int((y == 0).sum()))
print("Prevalencia:", round(100 * y.mean(), 4), "%")
print("Valores ausentes:", int(X.isna().sum().sum()))
print("IDs unicos:", ids.nunique())

print("\nVariaveis utilizadas:")
print(X.columns.tolist())

print("\nDistribuicao das variaveis:")
display(resumo_variaveis)

print("\nCorrelacao entre os preditores:")
display(X.corr().round(4))


### Stratified partition


In [ ]:
from sklearn.model_selection import train_test_split

df_modelagem = (
    df_analise
    .sort_values("EPOCH", kind="quicksort")
    .reset_index(drop=True)
)

X = df_modelagem[variaveis_modelo].copy()
y = df_modelagem[variavel_resposta].astype(int).copy()

indices = np.arange(len(df_modelagem))

indices_treino, indices_teste = train_test_split(
    indices,
    test_size=0.25,
    random_state=42,
    stratify=y
)

X_train = X.iloc[indices_treino].copy()
X_test = X.iloc[indices_teste].copy()

y_train = y.iloc[indices_treino].copy()
y_test = y.iloc[indices_teste].copy()

ids_treino = set(
    df_modelagem.iloc[indices_treino]["NORAD_CAT_ID"]
)

ids_teste = set(
    df_modelagem.iloc[indices_teste]["NORAD_CAT_ID"]
)

assert len(ids_treino & ids_teste) == 0
assert len(ids_treino | ids_teste) == len(df_modelagem)
assert len(X_train) == 9604
assert len(X_test) == 3202
assert y_train.sum() + y_test.sum() == 403

df_particoes = df_modelagem[
    [
        "NORAD_CAT_ID",
        "EPOCH",
        "OBSERVED_CATALOG_DECAY",
        "GRUPO_CONSTELACAO"
    ]
].copy()

df_particoes["PARTICAO"] = "treino"
df_particoes.loc[indices_teste, "PARTICAO"] = "teste"

arquivo_particoes = (
    diretorio_resultados / "particoes_treino_teste_v8.csv"
)

df_particoes.to_csv(arquivo_particoes, index=False)

print("Treinamento:", len(X_train))
print("Eventos no treinamento:", int(y_train.sum()))
print(
    "Prevalencia no treinamento:",
    round(100 * y_train.mean(), 4), "%"
)

print("\nTeste:", len(X_test))
print("Eventos no teste:", int(y_test.sum()))
print(
    "Prevalencia no teste:",
    round(100 * y_test.mean(), 4), "%"
)

print("\nIDs compartilhados:", len(ids_treino & ids_teste))
print("Arquivo salvo:", arquivo_particoes)

arquivo_previsoes_antigas = (
    diretorio_base
    / "revisao_jbcs_2026"
    / "tables"
    / "observed_decay_test_predictions.csv"
)

if arquivo_previsoes_antigas.is_file():
    previsoes_antigas = pd.read_csv(
        arquivo_previsoes_antigas
    )

    ids_antigos = set(
        previsoes_antigas["NORAD_CAT_ID"]
    )

    print("\nComparacao com a validacao anterior:")
    print("IDs em comum:", len(ids_teste & ids_antigos))
    print("IDs diferentes:", len(ids_teste ^ ids_antigos))
    print("Divisao reproduzida:", ids_teste == ids_antigos)

    if ids_teste != ids_antigos:
        print("A divisao ainda nao coincide com a analise anterior.")
else:
    print("\nArquivo de previsoes anteriores nao encontrado.")


### Random Forest


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss
)
import joblib

modelo_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("modelo", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ))
])

modelo_rf.fit(X_train, y_train)

classes = modelo_rf.named_steps["modelo"].classes_
indice_positivo = int(np.flatnonzero(classes == 1)[0])

probabilidades_rf = modelo_rf.predict_proba(X_test)[
    :, indice_positivo
]

ap_rf = average_precision_score(
    y_test, probabilidades_rf
)

auc_rf = roc_auc_score(
    y_test, probabilidades_rf
)

brier_rf = brier_score_loss(
    y_test, probabilidades_rf
)

df_predicoes_rf = df_modelagem.iloc[
    indices_teste
][
    [
        "NORAD_CAT_ID",
        "EPOCH",
        "PERIGEE_ALT_KM",
        "GRUPO_CONSTELACAO",
        "OBSERVED_CATALOG_DECAY"
    ]
].copy()

df_predicoes_rf["RF_PROBABILITY"] = probabilidades_rf

arquivo_predicoes = (
    diretorio_resultados / "predicoes_rf_180d_v8.csv"
)

arquivo_modelo = (
    diretorio_resultados / "random_forest_180d_v8.joblib"
)

df_predicoes_rf.to_csv(
    arquivo_predicoes, index=False
)

joblib.dump(modelo_rf, arquivo_modelo)

print("Treinamento:", len(X_train))
print("Teste:", len(X_test))
print("Eventos no teste:", int(y_test.sum()))

print("\nDesempenho do Random Forest:")
print(f"Average Precision: {ap_rf:.7f}")
print(f"ROC-AUC: {auc_rf:.7f}")
print(f"Brier Score: {brier_rf:.7f}")

print("\nComparacao com as previsoes anteriores:")

if arquivo_previsoes_antigas.is_file():
    anteriores = pd.read_csv(
        arquivo_previsoes_antigas
    )

    comparacao = df_predicoes_rf.merge(
        anteriores[
            [
                "NORAD_CAT_ID",
                "observed",
                "rf_p_decay"
            ]
        ],
        on="NORAD_CAT_ID",
        how="inner",
        validate="one_to_one"
    )

    eventos_divergentes = int((
        comparacao["OBSERVED_CATALOG_DECAY"]
        != comparacao["observed"]
    ).sum())

    diferencas = np.abs(
        comparacao["RF_PROBABILITY"]
        - comparacao["rf_p_decay"]
    )

    print("Objetos comparados:", len(comparacao))
    print("Desfechos divergentes:", eventos_divergentes)
    print(
        "Diferenca maxima entre probabilidades:",
        f"{diferencas.max():.10f}"
    )
    print(
        "Diferenca media entre probabilidades:",
        f"{diferencas.mean():.10f}"
    )

    if len(comparacao) != len(X_test):
        raise ValueError("Nem todos os objetos foram comparados.")

    if eventos_divergentes:
        raise ValueError("Foram encontrados desfechos divergentes.")

else:
    print("Arquivo anterior nao encontrado.")

print("\nArquivos salvos:")
print(arquivo_predicoes)
print(arquivo_modelo)


### Perigee ranking baseline


In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

escore_perigeu = -X_test["PERIGEE_ALT_KM"].to_numpy(dtype=float)

ap_perigeu = average_precision_score(
    y_test, escore_perigeu
)

auc_perigeu = roc_auc_score(
    y_test, escore_perigeu
)

prevalencia_teste = y_test.mean()

comparacao_modelos = pd.DataFrame({
    "Modelo": [
        "Random Forest",
        "Perigeu isolado"
    ],
    "Average_Precision": [
        ap_rf,
        ap_perigeu
    ],
    "ROC_AUC": [
        auc_rf,
        auc_perigeu
    ]
})

df_predicoes_comparadas = df_predicoes_rf.copy()

df_predicoes_comparadas["PERIGEE_SCORE"] = escore_perigeu

arquivo_comparacao = (
    diretorio_resultados / "comparacao_rf_perigeu_v8.csv"
)

arquivo_predicoes_comparadas = (
    diretorio_resultados / "predicoes_comparadas_v8.csv"
)

comparacao_modelos.to_csv(
    arquivo_comparacao, index=False
)

df_predicoes_comparadas.to_csv(
    arquivo_predicoes_comparadas, index=False
)

print("Comparacao dos metodos:")
display(comparacao_modelos.round(7))

print(f"\nPrevalencia no teste: {prevalencia_teste:.7f}")
print(f"Diferenca de Average Precision: {ap_rf - ap_perigeu:.7f}")
print(f"Diferenca de ROC-AUC: {auc_rf - auc_perigeu:.7f}")

if arquivo_previsoes_antigas.is_file():
    anteriores = pd.read_csv(arquivo_previsoes_antigas)

    verificacao = df_predicoes_comparadas.merge(
        anteriores[
            ["NORAD_CAT_ID", "perigee_score"]
        ],
        on="NORAD_CAT_ID",
        how="inner",
        validate="one_to_one"
    )

    if len(verificacao) != len(X_test):
        raise ValueError("A comparacao nao incluiu todos os objetos.")

    diferenca_perigeu = np.abs(
        verificacao["PERIGEE_SCORE"]
        - verificacao["perigee_score"]
    )

    print("\nComparacao com a analise anterior:")
    print("Objetos comparados:", len(verificacao))
    print(
        "Diferenca maxima entre escores:",
        f"{diferenca_perigeu.max():.10f}"
    )
    print(
        "Escores reproduzidos:",
        bool(np.allclose(
            verificacao["PERIGEE_SCORE"],
            verificacao["perigee_score"],
            rtol=0,
            atol=1e-8
        ))
    )

print("\nArquivos salvos:")
print(arquivo_comparacao)
print(arquivo_predicoes_comparadas)


### Bootstrap and calibration


In [ ]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss
)

y_observado = y_test.to_numpy(dtype=int)
p_rf = np.asarray(probabilidades_rf, dtype=float)
s_perigeu = np.asarray(escore_perigeu, dtype=float)

if not (
    len(y_observado) == len(p_rf) == len(s_perigeu)
):
    raise ValueError("Os vetores possuem tamanhos diferentes.")

if not np.isfinite(p_rf).all():
    raise ValueError("Probabilidades nao finitas.")

if ((p_rf < 0) | (p_rf > 1)).any():
    raise ValueError("Probabilidades fora do intervalo [0, 1].")

indices_positivos = np.flatnonzero(y_observado == 1)
indices_negativos = np.flatnonzero(y_observado == 0)

n_bootstrap = 2000
gerador = np.random.default_rng(42)

resultados_bootstrap = np.empty((n_bootstrap, 5))

for i in range(n_bootstrap):
    positivos = gerador.choice(
        indices_positivos,
        size=len(indices_positivos),
        replace=True
    )

    negativos = gerador.choice(
        indices_negativos,
        size=len(indices_negativos),
        replace=True
    )

    amostra = np.concatenate([positivos, negativos])

    y_b = y_observado[amostra]
    rf_b = p_rf[amostra]
    perigeu_b = s_perigeu[amostra]

    resultados_bootstrap[i, 0] = average_precision_score(
        y_b, rf_b
    )

    resultados_bootstrap[i, 1] = average_precision_score(
        y_b, perigeu_b
    )

    resultados_bootstrap[i, 2] = roc_auc_score(
        y_b, rf_b
    )

    resultados_bootstrap[i, 3] = roc_auc_score(
        y_b, perigeu_b
    )

    resultados_bootstrap[i, 4] = brier_score_loss(
        y_b, rf_b
    )

colunas_bootstrap = [
    "AP_RF",
    "AP_PERIGEE",
    "AUC_RF",
    "AUC_PERIGEE",
    "BRIER_RF"
]

df_bootstrap = pd.DataFrame(
    resultados_bootstrap,
    columns=colunas_bootstrap
)

df_bootstrap["DELTA_AP"] = (
    df_bootstrap["AP_RF"] - df_bootstrap["AP_PERIGEE"]
)

df_bootstrap["DELTA_AUC"] = (
    df_bootstrap["AUC_RF"] - df_bootstrap["AUC_PERIGEE"]
)

estimativas = {
    "AP_RF": ap_rf,
    "AP_PERIGEE": ap_perigeu,
    "AUC_RF": auc_rf,
    "AUC_PERIGEE": auc_perigeu,
    "BRIER_RF": brier_rf,
    "DELTA_AP": ap_rf - ap_perigeu,
    "DELTA_AUC": auc_rf - auc_perigeu
}

resumo_ic = []

for metrica, estimativa in estimativas.items():
    inferior, superior = np.percentile(
        df_bootstrap[metrica],
        [2.5, 97.5]
    )

    resumo_ic.append({
        "metrica": metrica,
        "estimativa": estimativa,
        "IC95_inferior": inferior,
        "IC95_superior": superior
    })

tabela_ic = pd.DataFrame(resumo_ic)

probabilidade_referencia = y_train.mean()

brier_referencia = brier_score_loss(
    y_observado,
    np.full(len(y_observado), probabilidade_referencia)
)

df_calibracao = pd.DataFrame({
    "observado": y_observado,
    "probabilidade": p_rf
})

df_calibracao["faixa_probabilidade"] = pd.qcut(
    df_calibracao["probabilidade"],
    q=10,
    duplicates="drop"
)

tabela_calibracao = (
    df_calibracao
    .groupby("faixa_probabilidade", observed=True)
    .agg(
        objetos=("observado", "size"),
        eventos=("observado", "sum"),
        probabilidade_media=("probabilidade", "mean"),
        frequencia_observada=("observado", "mean")
    )
    .reset_index()
)

tabela_calibracao["faixa_probabilidade"] = (
    tabela_calibracao["faixa_probabilidade"].astype(str)
)

df_bootstrap.to_csv(
    diretorio_resultados / "bootstrap_metricas_v8.csv",
    index=False
)

tabela_ic.to_csv(
    diretorio_resultados / "intervalos_confianca_v8.csv",
    index=False
)

tabela_calibracao.to_csv(
    diretorio_resultados / "calibracao_rf_v8.csv",
    index=False
)

print("Reamostragens:", n_bootstrap)
print("Eventos positivos no teste:", len(indices_positivos))
print("Eventos negativos no teste:", len(indices_negativos))

print("\nMetricas e intervalos de confianca de 95%:")
display(tabela_ic.round(6))

print("\nCalibracao do Random Forest:")
display(tabela_calibracao.round(6))

print("\nProbabilidade media prevista:", round(p_rf.mean(), 6))
print("Frequencia observada:", round(y_observado.mean(), 6))
print("Brier Score do Random Forest:", round(brier_rf, 6))
print("Brier Score da referencia constante:", round(brier_referencia, 6))

print("\nArquivos salvos em:", diretorio_resultados)


### Subgroup evaluation


In [ ]:
from sklearn.metrics import brier_score_loss, average_precision_score, roc_auc_score


df_teste_avaliacao = df_predicoes_comparadas.copy()

if df_teste_avaliacao["NORAD_CAT_ID"].duplicated().any():
    raise ValueError("Existem objetos duplicados no conjunto de teste.")

if len(df_teste_avaliacao) != len(y_test):
    raise ValueError("O numero de previsoes esta incorreto.")

def avaliar_subgrupo(nome, dados):
    observado = dados["OBSERVED_CATALOG_DECAY"].to_numpy(dtype=int)
    prob_rf = dados["RF_PROBABILITY"].to_numpy(dtype=float)
    score_perigeu = dados["PERIGEE_SCORE"].to_numpy(dtype=float)

    n = len(dados)
    eventos = int(observado.sum())

    resultado = {
        "grupo": nome,
        "objetos": n,
        "eventos": eventos,
        "prevalencia": eventos / n,
        "probabilidade_media_rf": prob_rf.mean(),
        "eventos_previstos_rf": prob_rf.sum(),
        "brier_rf": brier_score_loss(observado, prob_rf)
    }

    if 0 < eventos < n:
        resultado["ap_rf"] = average_precision_score(
            observado, prob_rf
        )
        resultado["ap_perigeu"] = average_precision_score(
            observado, score_perigeu
        )
        resultado["auc_rf"] = roc_auc_score(
            observado, prob_rf
        )
        resultado["auc_perigeu"] = roc_auc_score(
            observado, score_perigeu
        )
    else:
        resultado["ap_rf"] = np.nan
        resultado["ap_perigeu"] = np.nan
        resultado["auc_rf"] = np.nan
        resultado["auc_perigeu"] = np.nan

    return resultado


altitude = df_teste_avaliacao["PERIGEE_ALT_KM"]
starlink = df_teste_avaliacao["GRUPO_CONSTELACAO"].eq("Starlink")

subconjuntos = {
    "Teste completo": df_teste_avaliacao,
    "Perigeu <400 km": df_teste_avaliacao.loc[altitude < 400],
    "Perigeu 400-600 km": df_teste_avaliacao.loc[
        (altitude >= 400) & (altitude < 600)
    ],
    "Perigeu <600 km": df_teste_avaliacao.loc[altitude < 600],
    "Perigeu >=600 km": df_teste_avaliacao.loc[altitude >= 600],
    "Starlink": df_teste_avaliacao.loc[starlink],
    "Demais objetos": df_teste_avaliacao.loc[~starlink]
}

resultados_subgrupos = [
    avaliar_subgrupo(nome, dados)
    for nome, dados in subconjuntos.items()
]

tabela_subgrupos = pd.DataFrame(resultados_subgrupos)

tabela_subgrupos["delta_ap"] = (
    tabela_subgrupos["ap_rf"]
    - tabela_subgrupos["ap_perigeu"]
)

tabela_subgrupos["delta_auc"] = (
    tabela_subgrupos["auc_rf"]
    - tabela_subgrupos["auc_perigeu"]
)

arquivo_subgrupos = (
    diretorio_resultados / "desempenho_subgrupos_v8.csv"
)

tabela_subgrupos.to_csv(arquivo_subgrupos, index=False)

print("Desempenho por faixa orbital e populacao:")
display(
    tabela_subgrupos[
        [
            "grupo",
            "objetos",
            "eventos",
            "prevalencia",
            "ap_rf",
            "ap_perigeu",
            "auc_rf",
            "auc_perigeu",
            "delta_ap",
            "delta_auc"
        ]
    ].round(5)
)

print("\nProbabilidades previstas e eventos observados:")
display(
    tabela_subgrupos[
        [
            "grupo",
            "objetos",
            "eventos",
            "eventos_previstos_rf",
            "probabilidade_media_rf",
            "brier_rf"
        ]
    ].round(5)
)

print("\nArquivo salvo:", arquivo_subgrupos)


### Population transfer


In [ ]:
from sklearn.base import clone
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss
)

df_transferencia = df_modelagem.copy()

mascara_starlink = (
    df_transferencia["GRUPO_CONSTELACAO"] == "Starlink"
)

experimentos = [
    ("Starlink para demais", mascara_starlink),
    ("Demais para Starlink", ~mascara_starlink)
]

resultados_transferencia = []

for nome, mascara_treino in experimentos:
    mascara_teste = ~mascara_treino

    X_origem = df_transferencia.loc[
        mascara_treino, variaveis_modelo
    ]

    y_origem = df_transferencia.loc[
        mascara_treino, variavel_resposta
    ].astype(int)

    X_destino = df_transferencia.loc[
        mascara_teste, variaveis_modelo
    ]

    y_destino = df_transferencia.loc[
        mascara_teste, variavel_resposta
    ].astype(int)

    ids_origem = set(
        df_transferencia.loc[mascara_treino, "NORAD_CAT_ID"]
    )

    ids_destino = set(
        df_transferencia.loc[mascara_teste, "NORAD_CAT_ID"]
    )

    if ids_origem & ids_destino:
        raise ValueError("Identificadores compartilhados entre populacoes.")

    if y_origem.nunique() != 2 or y_destino.nunique() != 2:
        raise ValueError("Uma das populacoes nao contem ambas as classes.")

    modelo_transferencia = clone(modelo_rf)

    modelo_transferencia.fit(X_origem, y_origem)

    probabilidades = modelo_transferencia.predict_proba(
        X_destino
    )[:, 1]

    escore_altitude = (
        -X_destino["PERIGEE_ALT_KM"].to_numpy()
    )

    ap_modelo = average_precision_score(
        y_destino, probabilidades
    )

    ap_altitude = average_precision_score(
        y_destino, escore_altitude
    )

    auc_modelo = roc_auc_score(
        y_destino, probabilidades
    )

    auc_altitude = roc_auc_score(
        y_destino, escore_altitude
    )

    brier_modelo = brier_score_loss(
        y_destino, probabilidades
    )

    resultados_transferencia.append({
        "experimento": nome,
        "objetos_treino": len(X_origem),
        "eventos_treino": int(y_origem.sum()),
        "objetos_teste": len(X_destino),
        "eventos_teste": int(y_destino.sum()),
        "ap_rf": ap_modelo,
        "ap_perigeu": ap_altitude,
        "auc_rf": auc_modelo,
        "auc_perigeu": auc_altitude,
        "brier_rf": brier_modelo,
        "probabilidade_media_rf": probabilidades.mean(),
        "frequencia_observada": y_destino.mean(),
        "perigeu_mediano_treino": X_origem["PERIGEE_ALT_KM"].median(),
        "perigeu_mediano_teste": X_destino["PERIGEE_ALT_KM"].median()
    })

    identificador = (
        "starlink_para_demais"
        if nome == "Starlink para demais"
        else "demais_para_starlink"
    )

    previsoes = df_transferencia.loc[
        mascara_teste,
        [
            "NORAD_CAT_ID",
            "PERIGEE_ALT_KM",
            "GRUPO_CONSTELACAO",
            "OBSERVED_CATALOG_DECAY"
        ]
    ].copy()

    previsoes["RF_PROBABILITY"] = probabilidades
    previsoes["PERIGEE_SCORE"] = escore_altitude

    previsoes.to_csv(
        diretorio_resultados / f"predicoes_{identificador}_v8.csv",
        index=False
    )

    joblib.dump(
        modelo_transferencia,
        diretorio_resultados / f"modelo_{identificador}_v8.joblib"
    )

tabela_transferencia = pd.DataFrame(
    resultados_transferencia
)

tabela_transferencia["delta_ap"] = (
    tabela_transferencia["ap_rf"]
    - tabela_transferencia["ap_perigeu"]
)

tabela_transferencia["delta_auc"] = (
    tabela_transferencia["auc_rf"]
    - tabela_transferencia["auc_perigeu"]
)

tabela_transferencia.to_csv(
    diretorio_resultados / "validacao_entre_populacoes_v8.csv",
    index=False
)

print("Validacao entre populacoes:")
display(
    tabela_transferencia[
        [
            "experimento",
            "objetos_treino",
            "eventos_treino",
            "objetos_teste",
            "eventos_teste",
            "ap_rf",
            "ap_perigeu",
            "auc_rf",
            "auc_perigeu",
            "delta_ap",
            "delta_auc"
        ]
    ].round(5)
)

print("\nDistribuicoes orbitais e calibracao:")
display(
    tabela_transferencia[
        [
            "experimento",
            "perigeu_mediano_treino",
            "perigeu_mediano_teste",
            "probabilidade_media_rf",
            "frequencia_observada",
            "brier_rf"
        ]
    ].round(6)
)

print("\nArquivo salvo:")
print(diretorio_resultados / "validacao_entre_populacoes_v8.csv")


### K-Means


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score

df_clusters = df_orbital.copy()

df_clusters["LOG_PERIGEE_ALT"] = np.log1p(
    df_clusters["PERIGEE_ALT_KM"].clip(lower=0)
)

variaveis_clusters = [
    "LOG_PERIGEE_ALT",
    "INCLINATION",
    "ECCENTRICITY"
]

X_clusters = df_clusters[variaveis_clusters].copy()

if not np.isfinite(X_clusters.to_numpy()).all():
    raise ValueError("Foram encontrados valores invalidos.")

padronizador = StandardScaler()
X_clusters_padronizado = padronizador.fit_transform(X_clusters)

resultados_k = []
modelos_k = {}

for k in range(2, 7):
    modelo = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    rotulos = modelo.fit_predict(X_clusters_padronizado)

    silhouette = silhouette_score(
        X_clusters_padronizado,
        rotulos,
        sample_size=1800,
        random_state=42
    )

    resultados_k.append({
        "k": k,
        "inercia": modelo.inertia_,
        "silhouette": silhouette
    })

    modelos_k[k] = modelo

tabela_selecao_k = pd.DataFrame(resultados_k)

modelo_kmeans = modelos_k[4]
rotulos_kmeans = modelo_kmeans.labels_

df_clusters["CLUSTER_SEM_RAAN"] = rotulos_kmeans

tabela_perfis_kmeans = (
    df_clusters
    .groupby("CLUSTER_SEM_RAAN")
    .agg(
        quantidade=("NORAD_CAT_ID", "size"),
        perigeu_mediano=("PERIGEE_ALT_KM", "median"),
        inclinacao_mediana=("INCLINATION", "median"),
        excentricidade_mediana=("ECCENTRICITY", "median")
    )
    .reset_index()
)

tabela_perfis_kmeans["percentual"] = (
    100 * tabela_perfis_kmeans["quantidade"] / len(df_clusters)
)

resultados_estabilidade = []

for semente in [0, 1, 10, 21, 43, 99]:
    modelo_alternativo = KMeans(
        n_clusters=4,
        random_state=semente,
        n_init=10
    )

    rotulos_alternativos = modelo_alternativo.fit_predict(
        X_clusters_padronizado
    )

    resultados_estabilidade.append({
        "semente": semente,
        "ARI": adjusted_rand_score(
            rotulos_kmeans,
            rotulos_alternativos
        )
    })

tabela_estabilidade = pd.DataFrame(resultados_estabilidade)

atribuicoes_kmeans = df_clusters[
    [
        "NORAD_CAT_ID",
        "OBJECT_NAME",
        "PERIGEE_ALT_KM",
        "INCLINATION",
        "ECCENTRICITY",
        "CLUSTER_SEM_RAAN"
    ]
].copy()

assert len(df_clusters) == 12810
assert tabela_perfis_kmeans["quantidade"].sum() == 12810
assert len(tabela_perfis_kmeans) == 4

tabela_selecao_k.to_csv(
    diretorio_resultados / "kmeans_selecao_k_v8.csv",
    index=False
)

tabela_perfis_kmeans.to_csv(
    diretorio_resultados / "kmeans_perfis_v8.csv",
    index=False
)

tabela_estabilidade.to_csv(
    diretorio_resultados / "kmeans_estabilidade_v8.csv",
    index=False
)

atribuicoes_kmeans.to_csv(
    diretorio_resultados / "kmeans_atribuicoes_v8.csv",
    index=False
)

joblib.dump(
    {
        "padronizador": padronizador,
        "modelo": modelo_kmeans,
        "variaveis": variaveis_clusters
    },
    diretorio_resultados / "kmeans_sem_raan_v8.joblib"
)

print("Objetos analisados:", len(df_clusters))
print("Variaveis:", variaveis_clusters)

print("\nSelecao do numero de clusters:")
display(tabela_selecao_k.round(6))

print("\nCaracteristicas dos quatro clusters:")
display(tabela_perfis_kmeans.round(6))

print("\nEstabilidade entre sementes:")
display(tabela_estabilidade.round(6))

arquivo_k_anterior = (
    diretorio_base
    / "revisao_jbcs_2026"
    / "tables"
    / "kmeans_sem_raan_selecao_k.csv"
)

if arquivo_k_anterior.is_file():
    referencia = pd.read_csv(arquivo_k_anterior)

    verificacao = tabela_selecao_k.merge(
        referencia,
        on="k",
        suffixes=("_v8", "_anterior"),
        validate="one_to_one"
    )

    print("\nComparacao com a analise anterior:")
    print(
        "Diferenca maxima na inercia:",
        np.abs(
            verificacao["inercia_v8"]
            - verificacao["inercia_anterior"]
        ).max()
    )
    print(
        "Diferenca maxima no silhouette:",
        np.abs(
            verificacao["silhouette_v8"]
            - verificacao["silhouette_anterior"]
        ).max()
    )

print("\nResultados salvos em:", diretorio_resultados)


### Isolation Forest


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import joblib

df_anomalias = df_orbital.copy()

df_anomalias["LOG_ABS_BSTAR"] = np.log10(
    df_anomalias["ABS_BSTAR"].clip(lower=1e-10)
)

variaveis_if = [
    "PERIGEE_ALT_KM",
    "LOG_ABS_BSTAR",
    "ECCENTRICITY",
    "MEAN_MOTION_DOT"
]

X_if = df_anomalias[variaveis_if].astype(float)

if not np.isfinite(X_if.to_numpy()).all():
    raise ValueError("Existem valores invalidos nas variaveis.")

padronizador_if = StandardScaler()
X_if_padronizado = padronizador_if.fit_transform(X_if)

escores_if = {}
modelos_if = {}

for semente in [42, 43, 44]:
    modelo = IsolationForest(
        n_estimators=200,
        max_samples="auto",
        contamination="auto",
        random_state=semente,
        n_jobs=-1
    )

    modelo.fit(X_if_padronizado)

    modelos_if[semente] = modelo
    escores_if[semente] = modelo.score_samples(
        X_if_padronizado
    )

df_anomalias["IF_SCORE"] = escores_if[42]

df_anomalias["IF_PERCENTILE"] = (
    df_anomalias["IF_SCORE"].rank(pct=True)
)

resumo_limites = []

for percentual in [5, 10, 15]:
    coluna = f"IF_TOP_{percentual}PCT"

    df_anomalias[coluna] = (
        df_anomalias["IF_PERCENTILE"] <= percentual / 100
    ).astype(int)

    resumo_limites.append({
        "percentual": percentual,
        "objetos_selecionados": int(df_anomalias[coluna].sum()),
        "perigeu_mediano": df_anomalias.loc[
            df_anomalias[coluna] == 1,
            "PERIGEE_ALT_KM"
        ].median()
    })

tabela_limites_if = pd.DataFrame(resumo_limites)

resultados_estabilidade_if = []

for percentual in [5, 10, 15]:
    quantidade = max(
        1, round(len(df_anomalias) * percentual / 100)
    )

    referencia = set(
        np.argsort(escores_if[42])[:quantidade]
    )

    for semente in [43, 44]:
        comparacao = set(
            np.argsort(escores_if[semente])[:quantidade]
        )

        intersecao = len(referencia & comparacao)
        uniao = len(referencia | comparacao)

        resultados_estabilidade_if.append({
            "percentual": percentual,
            "semente_referencia": 42,
            "semente_comparacao": semente,
            "jaccard": intersecao / uniao
        })

tabela_estabilidade_if = pd.DataFrame(
    resultados_estabilidade_if
)

colunas_saida = [
    "NORAD_CAT_ID",
    "OBJECT_NAME",
    "PERIGEE_ALT_KM",
    "ABS_BSTAR",
    "ECCENTRICITY",
    "MEAN_MOTION_DOT",
    "IF_SCORE",
    "IF_PERCENTILE",
    "IF_TOP_5PCT",
    "IF_TOP_10PCT",
    "IF_TOP_15PCT"
]

df_anomalias[colunas_saida].to_csv(
    diretorio_resultados / "isolationforest_resultados_v8.csv",
    index=False
)

tabela_limites_if.to_csv(
    diretorio_resultados / "isolationforest_limites_v8.csv",
    index=False
)

tabela_estabilidade_if.to_csv(
    diretorio_resultados / "isolationforest_estabilidade_v8.csv",
    index=False
)

joblib.dump(
    {
        "modelo": modelos_if[42],
        "padronizador": padronizador_if,
        "variaveis": variaveis_if
    },
    diretorio_resultados / "isolationforest_v8.joblib"
)

print("Objetos analisados:", len(df_anomalias))
print("Variaveis utilizadas:", variaveis_if)

print("\nDistribuicao dos escores:")
print(df_anomalias["IF_SCORE"].describe().round(6))

print("\nSelecao por percentil:")
display(tabela_limites_if.round(4))

print("\nEstabilidade entre sementes:")
display(tabela_estabilidade_if.round(6))

arquivo_if_anterior = (
    diretorio_base
    / "revisao_jbcs_2026"
    / "tables"
    / "orbital_catalog_analyzed.csv"
)

if arquivo_if_anterior.is_file():
    dados_anteriores = pd.read_csv(
        arquivo_if_anterior,
        usecols=["NORAD_CAT_ID", "IF_SCORE"]
    )

    verificacao_if = df_anomalias[
        ["NORAD_CAT_ID", "IF_SCORE"]
    ].merge(
        dados_anteriores,
        on="NORAD_CAT_ID",
        suffixes=("_v8", "_anterior"),
        validate="one_to_one"
    )

    diferenca_if = np.abs(
        verificacao_if["IF_SCORE_v8"]
        - verificacao_if["IF_SCORE_anterior"]
    )

    print("\nComparacao com a analise anterior:")
    print("Objetos comparados:", len(verificacao_if))
    print("Diferenca maxima:", f"{diferenca_if.max():.10f}")
    print("Diferenca media:", f"{diferenca_if.mean():.10f}")
else:
    print("\nArquivo anterior de escores nao encontrado.")

print("\nResultados salvos em:", diretorio_resultados)


### NRLMSIS fixed-Ap experiment


In [ ]:
import numpy as np
import pandas as pd
import pymsis
from importlib.metadata import version

data = np.array([np.datetime64("2025-10-01T12:00")])
longitudes = np.array([0.0])
latitudes = np.array([0.0])
altitudes = np.arange(200.0, 801.0, 25.0)

f107 = np.array([150.0])
f107a = np.array([150.0])

def calcular_densidade(altitude, ap):
    resultado = pymsis.calculate(
        dates=data,
        lons=longitudes,
        lats=latitudes,
        alts=np.array([float(altitude)]),
        f107s=f107,
        f107as=f107a,
        aps=np.full((1, 7), float(ap)),
        version=2.0,
        geomagnetic_activity=1
    )

    densidade = float(
        np.asarray(
            resultado[..., pymsis.Variable.MASS_DENSITY]
        ).reshape(-1)[0]
    )

    if not np.isfinite(densidade) or densidade <= 0:
        raise ValueError(
            f"Densidade invalida: altitude={altitude}, Ap={ap}"
        )

    return densidade

rho_calma = np.array([
    calcular_densidade(altitude, 4)
    for altitude in altitudes
])

rho_perturbada = np.array([
    calcular_densidade(altitude, 80)
    for altitude in altitudes
])

tabela_nrlmsis = pd.DataFrame({
    "altitude_km": altitudes,
    "rho_quiet_kg_m3": rho_calma,
    "rho_disturbed_kg_m3": rho_perturbada
})

tabela_nrlmsis["density_ratio"] = (
    tabela_nrlmsis["rho_disturbed_kg_m3"]
    / tabela_nrlmsis["rho_quiet_kg_m3"]
)

tabela_nrlmsis["relative_change_percent"] = (
    100 * (tabela_nrlmsis["density_ratio"] - 1)
)

print("Versao pymsis:", version("pymsis"))
print("Modelo: NRLMSIS 2.0")
print("Data:", data[0])
print("Coordenadas:", latitudes[0], longitudes[0])
print("F10.7 e F10.7a:", f107[0], f107a[0])
print("Ap:", 4, 80)
print("Altitudes calculadas:", len(tabela_nrlmsis))

altitudes_selecionadas = [
    200, 300, 400, 500, 550, 600, 700, 800
]

print("\nDensidades e variacoes relativas:")

display(
    tabela_nrlmsis.loc[
        tabela_nrlmsis["altitude_km"].isin(
            altitudes_selecionadas
        )
    ]
)

arquivo_anterior = (
    diretorio_base
    / "revisao_jbcs_2026"
    / "tables"
    / "nrlmsis_density_scenarios.csv"
)

if arquivo_anterior.is_file():
    anterior = pd.read_csv(arquivo_anterior)

    if "altitude_km" in anterior.columns:
        comparacao = tabela_nrlmsis.merge(
            anterior,
            on="altitude_km",
            suffixes=("_v8", "_anterior"),
            validate="one_to_one"
        )

        print("\nComparacao com resultados anteriores:")

        print("Altitudes coincidentes:", len(comparacao))

        for coluna in [
            "rho_quiet_kg_m3",
            "rho_disturbed_kg_m3",
            "relative_change_percent"
        ]:
            atual = f"{coluna}_v8"
            anterior_coluna = f"{coluna}_anterior"

            if (
                atual in comparacao.columns
                and anterior_coluna in comparacao.columns
            ):
                diferenca = np.abs(
                    comparacao[atual]
                    - comparacao[anterior_coluna]
                )

                print(
                    coluna,
                    "Diferenca maxima:",
                    f"{diferenca.max():.10e}"
                )
    else:
        print("\nTabela anterior sem coluna altitude_km.")
else:
    print("\nTabela anterior nao encontrada.")

arquivo_nrlmsis = (
    diretorio_resultados / "nrlmsis_density_scenarios_v8.csv"
)

tabela_nrlmsis.to_csv(
    arquivo_nrlmsis,
    index=False
)

print("\nArquivo salvo:", arquivo_nrlmsis)


### May 2024 storm


In [ ]:
datas_tempestade = np.arange(
    np.datetime64("2024-05-10T00:00"),
    np.datetime64("2024-05-13T00:00"),
    np.timedelta64(3, "h")
)

candidatos_indices = [
    diretorio_base / "indices_maio2024_v8.csv",
    diretorio_referencia / "indices_maio2024_v8.csv"
]
arquivos_indices = [p for p in candidatos_indices if p.is_file()]
if not arquivos_indices:
    raise FileNotFoundError(
        "Archived 2024 space-weather indices are required for an exact run. "
        "Copy indices_maio2024_v8.csv into the input data directory."
    )

arquivo_indices_referencia = arquivos_indices[0]
indices_salvos = pd.read_csv(arquivo_indices_referencia)
colunas_indices = ["data_utc", "f107", "f107a"] + [
    f"ap_{i}" for i in range(7)
]
if not set(colunas_indices).issubset(indices_salvos.columns):
    raise ValueError("The archived space-weather input is missing required columns.")
if len(indices_salvos) != len(datas_tempestade):
    raise ValueError("Expected 24 rows of three-hour space-weather inputs.")
datas_salvas = pd.to_datetime(
    indices_salvos["data_utc"], utc=True
).dt.tz_localize(None).to_numpy(dtype="datetime64[ns]")
if not np.array_equal(datas_salvas, datas_tempestade.astype("datetime64[ns]")):
    raise ValueError("Archived geomagnetic index dates do not match the experiment.")
f107_hist = indices_salvos["f107"].to_numpy(dtype=float)
f107a_hist = indices_salvos["f107a"].to_numpy(dtype=float)
ap_hist = indices_salvos[[f"ap_{i}" for i in range(7)]].to_numpy(dtype=float)
print("Using archived input:", arquivo_indices_referencia)
print("Space-weather SHA-256:", hashlib.sha256(
    arquivo_indices_referencia.read_bytes()
).hexdigest())

if ap_hist.shape != (len(datas_tempestade), 7):
    raise ValueError("Dimensao incorreta dos indices Ap.")

if not all(
    np.isfinite(valores).all()
    for valores in [f107_hist, f107a_hist, ap_hist]
):
    raise ValueError("Indices solares ou geomagneticos invalidos.")

tabela_indices = pd.DataFrame(
    ap_hist,
    columns=[f"ap_{i}" for i in range(7)]
)

tabela_indices.insert(
    0, "data_utc", pd.to_datetime(datas_tempestade)
)

tabela_indices["f107"] = f107_hist
tabela_indices["f107a"] = f107a_hist

longitudes_tempestade = [-120.0, 0.0, 120.0]
latitudes_tempestade = [-60.0, -30.0, 0.0, 30.0, 60.0]
altitudes_tempestade = [300.0, 400.0, 500.0, 550.0, 600.0, 700.0]

pontos = pd.MultiIndex.from_product(
    [
        range(len(datas_tempestade)),
        longitudes_tempestade,
        latitudes_tempestade,
        altitudes_tempestade
    ],
    names=[
        "indice_tempo",
        "longitude",
        "latitude",
        "altitude_km"
    ]
).to_frame(index=False)

indice_tempo = pontos["indice_tempo"].to_numpy(dtype=int)

datas_calculo = datas_tempestade[indice_tempo]
f107_calculo = f107_hist[indice_tempo]
f107a_calculo = f107a_hist[indice_tempo]
ap_calculo = ap_hist[indice_tempo]

longitude_calculo = pontos["longitude"].to_numpy(dtype=float)
latitude_calculo = pontos["latitude"].to_numpy(dtype=float)
altitude_calculo = pontos["altitude_km"].to_numpy(dtype=float)

resultado_historico = pymsis.calculate(
    dates=datas_calculo,
    lons=longitude_calculo,
    lats=latitude_calculo,
    alts=altitude_calculo,
    f107s=f107_calculo,
    f107as=f107a_calculo,
    aps=ap_calculo,
    version=2.0,
    geomagnetic_activity=-1
)

rho_historica = np.asarray(
    resultado_historico[..., pymsis.Variable.MASS_DENSITY],
    dtype=float
).reshape(-1)

if (
    len(rho_historica) != len(pontos)
    or not np.isfinite(rho_historica).all()
    or (rho_historica <= 0).any()
):
    raise ValueError("Densidades historicas invalidas.")

def densidade_referencia(i):
    resultado = pymsis.calculate(
        dates=datas_calculo[i:i + 1],
        lons=np.array([longitude_calculo[i]]),
        lats=np.array([latitude_calculo[i]]),
        alts=np.array([altitude_calculo[i]]),
        f107s=np.array([f107_calculo[i]]),
        f107as=np.array([f107a_calculo[i]]),
        aps=np.full((1, 7), 4.0),
        version=2.0,
        geomagnetic_activity=-1
    )

    densidade = float(
        np.asarray(
            resultado[..., pymsis.Variable.MASS_DENSITY]
        ).reshape(-1)[0]
    )

    if not np.isfinite(densidade) or densidade <= 0:
        raise ValueError(
            f"Densidade de referencia invalida no ponto {i}."
        )

    return densidade

rho_referencia = np.empty(len(pontos), dtype=float)

for i in range(len(pontos)):
    rho_referencia[i] = densidade_referencia(i)

    if (i + 1) % 540 == 0:
        print("Pontos de referencia calculados:", i + 1)

pontos["data_utc"] = pd.to_datetime(datas_calculo)
pontos["rho_historica_kg_m3"] = rho_historica
pontos["rho_referencia_kg_m3"] = rho_referencia

pontos["variacao_percentual"] = (
    100.0 * (
        pontos["rho_historica_kg_m3"]
        / pontos["rho_referencia_kg_m3"]
        - 1.0
    )
)

tabela_tempestade = (
    pontos.groupby("altitude_km")["variacao_percentual"]
    .agg(
        mediana_percent="median",
        media_percent="mean",
        minimo_percent="min",
        maximo_percent="max"
    )
    .reset_index()
)

tabela_indices.to_csv(
    diretorio_resultados / "indices_maio2024_v8.csv",
    index=False
)

pontos.to_csv(
    diretorio_resultados / "nrlmsis_maio2024_pontos_v8.csv",
    index=False
)

tabela_tempestade.to_csv(
    diretorio_resultados / "nrlmsis_tempestade_maio2024_v8.csv",
    index=False
)

print("\nInstantes analisados:", len(datas_tempestade))
print("Pontos por cenario:", len(pontos))

print("\nIndices historicos:")
display(
    tabela_indices[
        ["data_utc", "f107", "f107a", "ap_0", "ap_1"]
    ]
)

print("\nVariacao percentual da densidade:")
display(tabela_tempestade.round(6))

arquivo_anterior = (
    diretorio_base
    / "revisao_jbcs_2026"
    / "tables"
    / "nrlmsis_tempestade_maio2024.csv"
)

if arquivo_anterior.is_file():
    anterior = pd.read_csv(arquivo_anterior)

    colunas_comparacao = {
        "altitude_km",
        "mediana_percent",
        "maximo_percent"
    }

    if colunas_comparacao.issubset(anterior.columns):
        comparacao = tabela_tempestade.merge(
            anterior,
            on="altitude_km",
            suffixes=("_v8", "_anterior"),
            validate="one_to_one"
        )

        print("\nComparacao com a analise anterior:")
        display(
            comparacao[
                [
                    "altitude_km",
                    "mediana_percent_v8",
                    "mediana_percent_anterior",
                    "maximo_percent_v8",
                    "maximo_percent_anterior"
                ]
            ].round(6)
        )
    else:
        print("\nColunas da tabela anterior:")
        print(anterior.columns.tolist())
else:
    print("\nTabela anterior nao encontrada.")

print("\nResultados salvos em:", diretorio_resultados)


### Batch/point consistency


In [ ]:
indices_amostra = np.linspace(
    0, len(pontos) - 1, 16, dtype=int
)

pontos_550 = pontos.loc[
    pontos["altitude_km"] == 550.0
]

indices_extremos = np.concatenate([
    pontos_550.nlargest(4, "variacao_percentual").index.to_numpy(),
    pontos_550.nsmallest(4, "variacao_percentual").index.to_numpy()
])

indices_verificacao = np.unique(
    np.concatenate([indices_amostra, indices_extremos])
)

resultados_verificacao = []

for i in indices_verificacao:
    resultado = pymsis.calculate(
        dates=datas_calculo[i:i + 1],
        lons=np.array([longitude_calculo[i]]),
        lats=np.array([latitude_calculo[i]]),
        alts=np.array([altitude_calculo[i]]),
        f107s=np.array([f107_calculo[i]]),
        f107as=np.array([f107a_calculo[i]]),
        aps=ap_calculo[i:i + 1].copy(),
        version=2.0,
        geomagnetic_activity=-1
    )

    densidade_individual = float(
        np.asarray(
            resultado[..., pymsis.Variable.MASS_DENSITY]
        ).reshape(-1)[0]
    )

    densidade_lote = float(
        pontos.loc[i, "rho_historica_kg_m3"]
    )

    diferenca_relativa = (
        100 * (densidade_individual / densidade_lote - 1)
    )

    resultados_verificacao.append({
        "indice": i,
        "data_utc": datas_calculo[i],
        "latitude": latitude_calculo[i],
        "longitude": longitude_calculo[i],
        "altitude_km": altitude_calculo[i],
        "rho_lote": densidade_lote,
        "rho_individual": densidade_individual,
        "diferenca_percentual": diferenca_relativa
    })

tabela_verificacao = pd.DataFrame(resultados_verificacao)

if not np.isfinite(
    tabela_verificacao[
        ["rho_lote", "rho_individual", "diferenca_percentual"]
    ].to_numpy()
).all():
    raise ValueError("Foram encontrados valores invalidos.")

diferenca_maxima = (
    tabela_verificacao["diferenca_percentual"].abs().max()
)

print("Pontos verificados:", len(tabela_verificacao))
print(
    "Diferenca relativa maxima (%):",
    f"{diferenca_maxima:.10f}"
)

if diferenca_maxima >= 0.001:
    raise RuntimeError("Batch and individual MSIS calculations disagree.")
print("Consistent pointwise MSIS calculations: True")

display(
    tabela_verificacao[
        [
            "data_utc",
            "latitude",
            "longitude",
            "altitude_km",
            "diferenca_percentual"
        ]
    ].round(8)
)


### WACCM-X file validation


In [ ]:
import hashlib
import xarray as xr
from importlib.metadata import version

arquivos_waccmx = sorted(
    diretorio_base.rglob("DEN-CO2scaling2000-2500_v2.nc")
)

if not arquivos_waccmx:
    raise FileNotFoundError(
        "Arquivo DEN-CO2scaling2000-2500_v2.nc nao encontrado "
        "no diretorio Aeroespacial."
    )

print("Arquivos encontrados:", len(arquivos_waccmx))

for arquivo in arquivos_waccmx:
    print(arquivo)

if len(arquivos_waccmx) != 1:
    raise ValueError("Multiple WACCM-X source files found; select one explicitly.")

arquivo_waccmx = arquivos_waccmx[0]

hash_waccmx = hashlib.sha256()

with open(arquivo_waccmx, "rb") as arquivo:
    for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):
        hash_waccmx.update(bloco)

if hash_waccmx.hexdigest() != hashes_esperados["WACCMX"]:
    raise ValueError("WACCM-X file checksum differs from the archived source.")

ds_waccmx = xr.open_dataset(arquivo_waccmx)

print("\nArquivo utilizado:", arquivo_waccmx)
print("Tamanho (bytes):", arquivo_waccmx.stat().st_size)
print("SHA-256:", hash_waccmx.hexdigest())
print("Versao xarray:", version("xarray"))

print("\nDimensoes:")
print(dict(ds_waccmx.sizes))

print("\nCoordenadas:")

for nome, coordenada in ds_waccmx.coords.items():
    valores = np.asarray(coordenada.values)

    print(
        nome,
        "dimensoes:", coordenada.dims,
        "quantidade:", valores.size
    )

    if valores.size <= 20:
        print("Valores:", valores)
    else:
        print("Primeiro:", valores.flat[0])
        print("Ultimo:", valores.flat[-1])

print("\nVariaveis:")

for nome, variavel in ds_waccmx.data_vars.items():
    print(
        nome,
        "dimensoes:", variavel.dims,
        "formato:", variavel.shape,
        "tipo:", variavel.dtype
    )

print("\nAtributos globais:")
print(ds_waccmx.attrs)

print("\nEstrutura do conjunto de dados:")
display(ds_waccmx)


### WACCM-X scenarios


In [ ]:
import numpy as np
import pandas as pd

cenarios_waccmx = {
    "SSP1-2.6": "SSP1_26",
    "SSP2-4.5": "SSP2_45",
    "SSP5-8.5": "SSP5_85"
}

anos_waccmx = [2025, 2050, 2080, 2100]
altitudes_waccmx = [300, 400, 500, 550, 600, 700]
f107_waccmx = 150.0

def obter_fator(cenario, ano, altitude):
    valor = ds_waccmx[cenario].interp(
        alt=float(altitude),
        F107=f107_waccmx,
        method="linear"
    ).sel(year=int(ano)).item()

    return float(valor)

resultados_waccmx = []

for nome_cenario, variavel in cenarios_waccmx.items():
    for altitude in altitudes_waccmx:
        fator_referencia = obter_fator(
            variavel, 2025, altitude
        )

        if not np.isfinite(fator_referencia):
            raise ValueError(
                f"Referencia invalida: {nome_cenario}, {altitude} km"
            )

        if fator_referencia <= 0:
            raise ValueError("Fator de referencia nao positivo.")

        for ano in anos_waccmx:
            fator_ano = obter_fator(
                variavel, ano, altitude
            )

            if np.isfinite(fator_ano):
                razao = fator_ano / fator_referencia
                reducao = 100.0 * (1.0 - razao)
                situacao = "Disponivel"
            else:
                razao = np.nan
                reducao = np.nan
                situacao = "Indisponivel"

            resultados_waccmx.append({
                "cenario": nome_cenario,
                "ano": ano,
                "altitude_km": altitude,
                "F107": f107_waccmx,
                "fator_2025": fator_referencia,
                "fator_ano": fator_ano,
                "razao_relativa_2025": razao,
                "reducao_percentual": reducao,
                "situacao": situacao
            })

tabela_waccmx = pd.DataFrame(resultados_waccmx)

assert len(tabela_waccmx) == 72

arquivo_waccmx_v8 = (
    diretorio_resultados / "waccmx_cenarios_v8.csv"
)

tabela_waccmx.to_csv(
    arquivo_waccmx_v8, index=False
)

print("Cenarios:", list(cenarios_waccmx))
print("Anos:", anos_waccmx)
print("Altitudes:", altitudes_waccmx)
print("F10.7:", f107_waccmx)
print("Registros:", len(tabela_waccmx))

print("\nResultados a 550 km:")

tabela_550 = tabela_waccmx.loc[
    tabela_waccmx["altitude_km"] == 550,
    [
        "cenario",
        "ano",
        "fator_2025",
        "fator_ano",
        "reducao_percentual",
        "situacao"
    ]
]

display(tabela_550.round(6))

print("\nDisponibilidade dos cenarios:")

display(
    tabela_waccmx.groupby(
        ["cenario", "situacao"]
    ).size().reset_index(name="quantidade")
)

arquivo_anterior_waccmx = (
    diretorio_base
    / "revisao_jbcs_2026"
    / "tables"
    / "waccmx_cenarios_corrigidos.csv"
)

if arquivo_anterior_waccmx.is_file():
    anterior = pd.read_csv(arquivo_anterior_waccmx)

    chaves = [
        "cenario",
        "ano",
        "altitude_km",
        "F107"
    ]

    comparacao_waccmx = tabela_waccmx.merge(
        anterior,
        on=chaves,
        how="outer",
        suffixes=("_v8", "_anterior"),
        indicator=True,
        validate="one_to_one"
    )

    print("\nComparacao com a analise anterior:")

    print(
        "Registros correspondentes:",
        int((comparacao_waccmx["_merge"] == "both").sum())
    )

    print(
        "Registros sem correspondencia:",
        int((comparacao_waccmx["_merge"] != "both").sum())
    )

    for coluna in [
        "fator_2025",
        "fator_ano",
        "razao_relativa_2025",
        "reducao_percentual"
    ]:
        atual = comparacao_waccmx[
            f"{coluna}_v8"
        ].to_numpy(dtype=float)

        antigo = comparacao_waccmx[
            f"{coluna}_anterior"
        ].to_numpy(dtype=float)

        diferentes = ~np.isclose(
            atual,
            antigo,
            rtol=1e-6,
            atol=1e-8,
            equal_nan=True
        )

        diferencas_absolutas = np.abs(atual - antigo)

        diferenca_maxima = (
            np.nanmax(diferencas_absolutas)
            if np.isfinite(diferencas_absolutas).any()
            else np.nan
        )

        print(f"\n{coluna}")
        print("Valores divergentes:", int(diferentes.sum()))
        print("Diferenca maxima:", diferenca_maxima)

    situacoes_iguais = (
        comparacao_waccmx["situacao_v8"]
        == comparacao_waccmx["situacao_anterior"]
    )

    print(
        "\nSituacoes de disponibilidade coincidentes:",
        int(situacoes_iguais.sum()),
        "de",
        len(comparacao_waccmx)
    )

else:
    print("\nArquivo anterior nao encontrado.")

print("\nArquivo salvo:", arquivo_waccmx_v8)


### Atmospheric figures


In [ ]:
import matplotlib.pyplot as plt

diretorio_figuras = diretorio_resultados / "figuras"
diretorio_figuras.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "legend.fontsize": 9,
    "savefig.dpi": 300
})

def salvar_figura(fig, nome):
    for extensao in ["png", "svg"]:
        fig.savefig(
            diretorio_figuras / f"{nome}.{extensao}",
            dpi=300,
            bbox_inches="tight"
        )

alt = tabela_nrlmsis["altitude_km"].to_numpy()
rho_calma = tabela_nrlmsis["rho_quiet_kg_m3"].to_numpy()
rho_perturbada = tabela_nrlmsis["rho_disturbed_kg_m3"].to_numpy()

fig, ax = plt.subplots(figsize=(6.5, 4.2))

ax.semilogy(alt, rho_calma, label="Quiet conditions (Ap = 4)")
ax.semilogy(
    alt,
    rho_perturbada,
    label="Disturbed conditions (Ap = 80)"
)

ax.set(
    xlabel="Altitude (km)",
    ylabel=r"Neutral mass density (kg m$^{-3}$)"
)
ax.grid(alpha=0.25)
ax.legend(frameon=False)

salvar_figura(fig, "nrlmsis_density_profiles_v8")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(6.5, 4.2))

alt = tabela_tempestade["altitude_km"].to_numpy()
mediana = tabela_tempestade["mediana_percent"].to_numpy()
minimo = tabela_tempestade["minimo_percent"].to_numpy()
maximo = tabela_tempestade["maximo_percent"].to_numpy()

linha, = ax.plot(
    alt,
    mediana,
    marker="o",
    linewidth=2,
    label="Median"
)

ax.fill_between(
    alt,
    minimo,
    maximo,
    color=linha.get_color(),
    alpha=0.18,
    label="Minimum-maximum"
)

ax.set(
    xlabel="Altitude (km)",
    ylabel="Density enhancement (%)"
)
ax.grid(alpha=0.25)
ax.legend(frameon=False)

salvar_figura(fig, "nrlmsis_may2024_v8")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(6.5, 4.2))

for cenario in ["SSP1-2.6", "SSP2-4.5", "SSP5-8.5"]:
    selecao = tabela_waccmx.loc[
        (tabela_waccmx["cenario"] == cenario)
        & (tabela_waccmx["altitude_km"] == 550)
    ].sort_values("ano")

    ax.plot(
        selecao["ano"],
        selecao["reducao_percentual"],
        marker="o",
        linewidth=2,
        label=cenario
    )

ax.set(
    xlabel="Year",
    ylabel="Density reduction relative to 2025 (%)",
    xticks=[2025, 2050, 2080, 2100]
)
ax.grid(alpha=0.25)
ax.legend(frameon=False)

salvar_figura(fig, "waccmx_550km_v8")
plt.show()
plt.close(fig)

print("Figuras salvas em:")
print(diretorio_figuras)

for arquivo in sorted(diretorio_figuras.iterdir()):
    print(arquivo.name)


### Output integrity


In [ ]:
from pathlib import Path
from PIL import Image
import xml.etree.ElementTree as ET

nomes_figuras = [
    "nrlmsis_density_profiles_v8",
    "nrlmsis_may2024_v8",
    "waccmx_550km_v8"
]

print("VERIFICACAO DAS FIGURAS\n")

for nome in nomes_figuras:
    arquivo_png = diretorio_figuras / f"{nome}.png"
    arquivo_svg = diretorio_figuras / f"{nome}.svg"

    if not arquivo_png.is_file():
        raise FileNotFoundError(arquivo_png)

    if not arquivo_svg.is_file():
        raise FileNotFoundError(arquivo_svg)

    with Image.open(arquivo_png) as imagem:
        dimensoes = imagem.size
        imagem.verify()

    ET.parse(arquivo_svg)

    print(nome)
    print("  PNG:", arquivo_png.stat().st_size, "bytes")
    print("  SVG:", arquivo_svg.stat().st_size, "bytes")
    print("  Dimensoes:", dimensoes)
    print("  Integridade: OK")

print("\nVERIFICACAO DOS RESULTADOS\n")

arquivos_resultados = [
    "nrlmsis_density_scenarios_v8.csv",
    "indices_maio2024_v8.csv",
    "nrlmsis_maio2024_pontos_v8.csv",
    "nrlmsis_tempestade_maio2024_v8.csv",
    "waccmx_cenarios_v8.csv"
]

for nome in arquivos_resultados:
    arquivo = diretorio_resultados / nome

    if not arquivo.is_file():
        raise FileNotFoundError(arquivo)

    tabela = pd.read_csv(arquivo)

    print(
        nome,
        "| linhas:", len(tabela),
        "| colunas:", len(tabela.columns)
    )


### Audit


In [ ]:
import json
import hashlib
from datetime import datetime, timezone
from importlib.metadata import version
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss
)

pasta_auditoria = diretorio_resultados

arquivos_obrigatorios = {
    "NRLMSIS": "nrlmsis_density_scenarios_v8.csv",
    "Indices_2024": "indices_maio2024_v8.csv",
    "Tempestade_pontos": "nrlmsis_maio2024_pontos_v8.csv",
    "Tempestade_resumo": "nrlmsis_tempestade_maio2024_v8.csv",
    "WACCMX": "waccmx_cenarios_v8.csv"
}

checagens = []
tabelas_auditoria = {}

def verificar(nome, condicao, detalhe):
    checagens.append({
        "verificacao": nome,
        "status": "OK" if bool(condicao) else "FALHA",
        "detalhe": str(detalhe)
    })

for nome, arquivo_nome in arquivos_obrigatorios.items():
    caminho = pasta_auditoria / arquivo_nome

    verificar(
        f"Arquivo {nome}",
        caminho.is_file(),
        caminho
    )

    if caminho.is_file():
        tabelas_auditoria[nome] = pd.read_csv(caminho)

if len(tabelas_auditoria) != len(arquivos_obrigatorios):
    raise FileNotFoundError(
        "Faltam arquivos obrigatorios da auditoria."
    )

nrl = tabelas_auditoria["NRLMSIS"]
indices = tabelas_auditoria["Indices_2024"]
temp_pontos = tabelas_auditoria["Tempestade_pontos"]
temp_resumo = tabelas_auditoria["Tempestade_resumo"]
waccm = tabelas_auditoria["WACCMX"]

verificar(
    "NRLMSIS: 25 altitudes",
    len(nrl) == 25,
    len(nrl)
)

verificar(
    "NRLMSIS: densidades validas",
    np.isfinite(
        nrl[["rho_quiet_kg_m3", "rho_disturbed_kg_m3"]]
        .to_numpy(dtype=float)
    ).all()
    and (
        nrl[["rho_quiet_kg_m3", "rho_disturbed_kg_m3"]]
        .to_numpy(dtype=float) > 0
    ).all(),
    "Densidades finitas e positivas"
)

verificar(
    "Tempestade: 24 instantes",
    len(indices) == 24,
    len(indices)
)

verificar(
    "Tempestade: 2160 pontos",
    len(temp_pontos) == 2160,
    len(temp_pontos)
)

verificar(
    "Tempestade: resultados finitos",
    np.isfinite(
        temp_pontos["variacao_percentual"].to_numpy(dtype=float)
    ).all(),
    "Variacoes percentuais finitas"
)

resumo_recalculado = (
    temp_pontos.groupby("altitude_km")["variacao_percentual"]
    .agg(
        mediana_percent="median",
        media_percent="mean",
        minimo_percent="min",
        maximo_percent="max"
    )
    .reset_index()
)

resumo_comparado = temp_resumo.merge(
    resumo_recalculado,
    on="altitude_km",
    suffixes=("_salvo", "_recalculado"),
    validate="one_to_one"
)

colunas_resumo = [
    "mediana_percent",
    "media_percent",
    "minimo_percent",
    "maximo_percent"
]

resumos_iguais = all(
    np.allclose(
        resumo_comparado[f"{coluna}_salvo"],
        resumo_comparado[f"{coluna}_recalculado"],
        rtol=1e-10,
        atol=1e-10
    )
    for coluna in colunas_resumo
)

verificar(
    "Tempestade: estatisticas reproduzidas",
    resumos_iguais and len(resumo_comparado) == 6,
    "Recalculadas diretamente dos 2160 pontos"
)

verificar(
    "Storm regression at 550 km",
    np.isclose(
        temp_resumo.loc[
            temp_resumo["altitude_km"] == 550.0,
            "mediana_percent"
        ].iloc[0],
        80.426869,
        rtol=0,
        atol=0.01
    ),
    "Median must agree with archived 2024 space-weather input"
)

verificar(
    "WACCM-X: 72 registros",
    len(waccm) == 72,
    len(waccm)
)

indisponiveis = waccm.loc[
    (waccm["cenario"] == "SSP5-8.5")
    & (waccm["ano"] == 2100)
]

verificar(
    "WACCM-X: SSP5-8.5 em 2100",
    len(indisponiveis) == 6
    and indisponiveis["fator_ano"].isna().all(),
    "Seis resultados indisponiveis, sem extrapolacao"
)

resultado_550 = waccm.loc[
    (waccm["cenario"] == "SSP2-4.5")
    & (waccm["ano"] == 2100)
    & (waccm["altitude_km"] == 550)
]

verificar(
    "WACCM-X: SSP2-4.5 em 2100, 550 km",
    len(resultado_550) == 1
    and np.isclose(
        resultado_550["reducao_percentual"].iloc[0],
        30.811302,
        atol=1e-5
    ),
    "Reducao relativa esperada: 30,811302%"
)

metricas_rf = {}

for nome in ["y_test", "probabilidades_rf", "df_coorte", "df_clusters", "df_anomalias"]:
    if nome not in globals():
        raise RuntimeError(f"Required analysis stage has not been executed: {nome}")

if "y_test" in globals() and "probabilidades_rf" in globals():
    y_verificacao = np.asarray(y_test, dtype=int)
    probabilidades_verificacao = np.asarray(
        probabilidades_rf, dtype=float
    )

    metricas_rf = {
        "objetos_teste": int(len(y_verificacao)),
        "eventos_teste": int(y_verificacao.sum()),
        "average_precision": float(
            average_precision_score(
                y_verificacao,
                probabilidades_verificacao
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                y_verificacao,
                probabilidades_verificacao
            )
        ),
        "brier": float(
            brier_score_loss(
                y_verificacao,
                probabilidades_verificacao
            )
        ),
        "prevalencia_observada": float(y_verificacao.mean()),
        "probabilidade_media_prevista": float(
            probabilidades_verificacao.mean()
        )
    }

    verificar(
        "Random Forest: conjunto de teste",
        metricas_rf["objetos_teste"] == 3202
        and metricas_rf["eventos_teste"] == 101,
        metricas_rf
    )

    verificar(
        "Random Forest: Average Precision",
        np.isclose(
            metricas_rf["average_precision"],
            0.7752288,
            atol=1e-6
        ),
        metricas_rf["average_precision"]
    )

    verificar(
        "Random Forest: ROC AUC",
        np.isclose(
            metricas_rf["roc_auc"],
            0.9349300,
            atol=1e-6
        ),
        metricas_rf["roc_auc"]
    )

if "df_coorte" in globals():
    verificar(
        "Coorte: tamanho e eventos",
        len(df_coorte) == 12806
        and int(
            df_coorte["OBSERVED_CATALOG_DECAY"].sum()
        ) == 403,
        "12806 objetos e 403 eventos"
    )

if "df_clusters" in globals():
    verificar(
        "Agrupamento: amostra original",
        len(df_clusters) == 12810,
        len(df_clusters)
    )

if "df_anomalias" in globals():
    verificar(
        "Deteccao de anomalias: amostra original",
        len(df_anomalias) == 12810,
        len(df_anomalias)
    )

arquivos_origem = {
    "TLE": diretorio_base / "active_satellites.csv",
    "SATCAT": diretorio_base / "satcat.csv",
    "WACCMX_NetCDF": arquivo_waccmx
}

hashes_origem = {}

for nome, caminho in arquivos_origem.items():
    if caminho.is_file():
        digest = hashlib.sha256()

        with open(caminho, "rb") as origem:
            for bloco in iter(
                lambda: origem.read(1024 * 1024),
                b""
            ):
                digest.update(bloco)

        hashes_origem[nome] = digest.hexdigest()

for nome, digest_esperado in [
    ("TLE", hashes_esperados["TLE"]),
    ("SATCAT", hashes_esperados["SATCAT"]),
    ("WACCMX_NetCDF", hashes_esperados["WACCMX"])
]:
    verificar(
        f"Input SHA-256: {nome}",
        hashes_origem.get(nome) == digest_esperado,
        hashes_origem.get(nome, "missing")
    )

checagens.append({
    "verificacao": "SATCAT snapshot provenance",
    "status": "OK" if satcat_data_confirmada else "PENDING",
    "detalhe": str(data_extracao_satcat.date())
})

verificar(
    "Figuras atmosfericas",
    all(
        (diretorio_figuras / f"{nome}.{extensao}").is_file()
        for nome in [
            "nrlmsis_density_profiles_v8",
            "nrlmsis_may2024_v8",
            "waccmx_550km_v8"
        ]
        for extensao in ["png", "svg"]
    ),
    "Tres figuras em PNG e SVG"
)

relatorio_auditoria = {
    "data_utc": datetime.now(timezone.utc).isoformat(),
    "versoes": {
        "numpy": version("numpy"),
        "pandas": version("pandas"),
        "scikit-learn": version("scikit-learn"),
        "pymsis": version("pymsis"),
        "xarray": version("xarray")
    },
    "sha256_fontes": hashes_origem,
    "metricas_random_forest": metricas_rf,
    "checagens": checagens,
    "observacoes_metodologicas": [
        "Os eventos representam datas de decaimento catalogadas no SATCAT.",
        "O teste aleatorio estratificado nao constitui validacao temporal.",
        "As probabilidades do Random Forest apresentam superestimacao media.",
        "A transferencia entre populacoes orbitais apresenta perda de desempenho.",
        "A simulacao de maio de 2024 utiliza historico Ap trihorario.",
        "As medianas da tempestade diferem da tabela da analise anterior.",
        "Os fatores WACCM-X nao representam diretamente tempo de vida orbital.",
        "Nao foram extrapolados os valores ausentes de SSP5-8.5 em 2100."
    ]
}

arquivo_json = pasta_auditoria / "auditoria_cientifica_v8.json"
arquivo_csv = pasta_auditoria / "auditoria_cientifica_v8.csv"

with open(arquivo_json, "w", encoding="utf-8") as saida:
    json.dump(
        relatorio_auditoria,
        saida,
        ensure_ascii=False,
        indent=2
    )

tabela_checagens = pd.DataFrame(checagens)
tabela_checagens.to_csv(arquivo_csv, index=False)

print("RESULTADO DA AUDITORIA\n")

display(tabela_checagens)

print("\nMETRICAS DO RANDOM FOREST")
print(json.dumps(metricas_rf, indent=2))

print("\nSHA-256 DAS FONTES")
print(json.dumps(hashes_origem, indent=2))

print("\nVerificacoes aprovadas:",
      int((tabela_checagens["status"] == "OK").sum()))
print("Verificacoes com falha:",
      int((tabela_checagens["status"] == "FALHA").sum()))

print("\nRelatorios salvos:")
print(arquivo_json)
print(arquivo_csv)

print("Pending provenance items:", int((tabela_checagens["status"] == "PENDING").sum()))


### Environment and file manifest


In [ ]:
import json
import platform
import hashlib
from datetime import datetime, timezone
from importlib.metadata import version

pasta_final = diretorio_resultados
arquivo_auditoria = pasta_final / "auditoria_cientifica_v8.json"
auditoria = json.loads(arquivo_auditoria.read_text(encoding="utf-8"))
if any(item["status"] == "FALHA" for item in auditoria["checagens"]):
    raise RuntimeError("A scientific audit check failed.")

metricas = auditoria["metricas_random_forest"]
tempestade = pd.read_csv(pasta_final / "nrlmsis_tempestade_maio2024_v8.csv")
waccmx = pd.read_csv(pasta_final / "waccmx_cenarios_v8.csv")
valor_550 = tempestade.loc[tempestade["altitude_km"] == 550].iloc[0]
reducao_2100 = waccmx.loc[
    (waccmx["cenario"] == "SSP2-4.5")
    & (waccmx["ano"] == 2100)
    & (waccmx["altitude_km"] == 550),
    "reducao_percentual"
].iloc[0]

timestamp = datetime.now(timezone.utc).isoformat()
readme = f"""# Analysis record

Generated (UTC): {timestamp}

Input SHA-256 values:
- TLE: {auditoria['sha256_fontes']['TLE']}
- SATCAT: {auditoria['sha256_fontes']['SATCAT']}
- WACCM-X: {auditoria['sha256_fontes']['WACCMX_NetCDF']}

Sample: 12,810 orbital records; 12,806 eligible records after excluding four
same-day decay cases; 403 SATCAT decay records within 180 days.
The SATCAT cutoff is {data_extracao_satcat.date()}; its provenance is
{'confirmed by the operator' if satcat_data_confirmada else 'NOT independently verified'}.

Random Forest: 200 trees, max depth 10, minimum leaf size 5, balanced
subsampling, seed 42. Stratified random split, 75% train / 25% test.
AP={metricas['average_precision']:.8f}; AUC={metricas['roc_auc']:.8f};
Brier={metricas['brier']:.8f}.
The RF probabilities are poorly calibrated and the random split is not
out-of-time validation. Cross-population transfer is materially weaker.

K-Means uses log clipped perigee altitude, inclination and eccentricity.
K=4 is an interpretability choice, not the best silhouette score.
Isolation Forest uses perigee, log absolute BSTAR, eccentricity and
mean-motion derivative. Top-5/10/15 percent are score thresholds, not
measured anomaly rates.

NRLMSIS 2.0 fixed-Ap case: 2025-10-01 12:00 UTC, lat/lon=0/0,
F10.7=F10.7a=150, Ap=4 versus Ap=80, 200-800 km by 25 km.
May 2024 storm: 24 UTC times (3-hour spacing), 15 locations and six
altitudes; historical F10.7 and 7-part Ap compared against Ap=4.
At 550 km, the storm median is {valor_550['mediana_percent']:.6f}% and
the maximum is {valor_550['maximo_percent']:.6f}%.
These medians do not exactly match the previous aggregate-only analysis.
For exact reproduction, retain the archived seven-component Ap inputs.

WACCM-X: SSP1-2.6, SSP2-4.5 and SSP5-8.5, reference year 2025,
F10.7=150, linear interpolation in altitude and F10.7.
SSP2-4.5 at 550 km for 2100: {reducao_2100:.6f}% density-factor reduction.
SSP5-8.5 at 2100 has no valid data and is not extrapolated.

Cataloged decay is not necessarily uncontrolled reentry. Neither density
scenario is a time-integrated orbital lifetime or ground-impact model.
"""
(pasta_final / "README_reprodutibilidade.md").write_text(readme, encoding="ascii")

bibliotecas = [
    "numpy", "pandas", "scipy", "scikit-learn", "matplotlib",
    "pymsis", "xarray", "h5netcdf", "joblib", "Pillow"
]
versoes = {}
for nome in bibliotecas:
    try:
        versoes[nome] = version(nome)
    except Exception:
        versoes[nome] = "not installed"
ambiente = {
    "timestamp_utc": timestamp,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "packages": versoes,
    "source_sha256": auditoria["sha256_fontes"],
    "satcat_snapshot_verified": satcat_data_confirmada
}
(pasta_final / "ambiente_reproducibilidade.json").write_text(
    json.dumps(ambiente, indent=2, ensure_ascii=True), encoding="ascii"
)

manifesto = []
for arquivo in sorted(pasta_final.rglob("*")):
    if not arquivo.is_file() or arquivo.name == "manifesto_arquivos_v8.csv":
        continue
    digest = hashlib.sha256()
    with arquivo.open("rb") as entrada:
        for bloco in iter(lambda: entrada.read(1024 * 1024), b""):
            digest.update(bloco)
    manifesto.append({
        "arquivo": str(arquivo.relative_to(pasta_final)),
        "tamanho_bytes": arquivo.stat().st_size,
        "sha256": digest.hexdigest()
    })
pd.DataFrame(manifesto).to_csv(
    pasta_final / "manifesto_arquivos_v8.csv", index=False
)
print("Scientific checks:", len(auditoria["checagens"]))
print("Pending provenance:", sum(x["status"] == "PENDING" for x in auditoria["checagens"]))
print("Files recorded:", len(manifesto))
print("Results directory:", pasta_final)
